In [2]:
import pandas as pd
import numpy as np
import folium
import os

# 1) Cargar tu CSV
df = pd.read_csv("edificios_10km_o_mas.csv")

# 2) Convertir a numérico (por si vienen como texto)
for c in ["latitud","longitud","lat_cp","lon_cp","distancia_km"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 3) Filtro de coordenadas "sensatas" (México aprox.)
#    lat: 14 a 33, lon: -118 a -86
df_ok = df[
    df["latitud"].between(14,33) &
    df["longitud"].between(-118,-86) &
    df["lat_cp"].between(14,33) &
    df["lon_cp"].between(-118,-86)
].copy()

# 4) Recalcular distancia (Haversine) para asegurar que sea real
R = 6371.0
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df_ok["dist_km"] = haversine(df_ok["latitud"], df_ok["longitud"], df_ok["lat_cp"], df_ok["lon_cp"])

# 5) Quedarnos SOLO con >= 10 km (lo que pide la tarea)
resultados = df_ok[df_ok["dist_km"] >= 10].copy()

print("Filas totales CSV:", len(df))
print("Filas con coords válidas MX:", len(df_ok))
print("Filas finales (>=10km):", len(resultados))

# 6) Crear carpeta de mapas
os.makedirs("mapas", exist_ok=True)

# 7) Generar un mapa por cada fila (con markers + línea + distancia)
for i, row in resultados.reset_index(drop=True).iterrows():
    lat_e, lon_e = row["latitud"], row["longitud"]
    lat_c, lon_c = row["lat_cp"], row["lon_cp"]
    cp = row["codigo_postal"]
    dist = round(row["dist_km"], 2)

    mapa = folium.Map(location=[lat_e, lon_e], zoom_start=12)

    folium.Marker(
        [lat_e, lon_e],
        popup=f"Edificio<br>CP: {cp}<br>Lat/Lon: {lat_e}, {lon_e}",
        icon=folium.Icon(color="blue")
    ).add_to(mapa)

    folium.Marker(
        [lat_c, lon_c],
        popup=f"CP/Colonia<br>CP: {cp}<br>Lat/Lon: {lat_c}, {lon_c}",
        icon=folium.Icon(color="red")
    ).add_to(mapa)

    folium.PolyLine(
        [(lat_e, lon_e), (lat_c, lon_c)],
        tooltip=f"Distancia: {dist} km",
        color="green",
        weight=4
    ).add_to(mapa)

    mapa.save(f"mapas/mapa_{i}.html")

# 8) Crear la "aplicación": index.html con tabla + links
with open("index.html", "w", encoding="utf-8") as f:
    f.write("<h2>Edificios con distancia ≥ 10 km</h2>")
    f.write("<p>(Mapas con Leaflet/OpenStreetMap vía Folium - tecnología libre)</p>")
    f.write("<table border='1' cellpadding='6' cellspacing='0'>")
    f.write("<tr><th>#</th><th>CP</th><th>Distancia (km)</th><th>Mapa</th></tr>")

    for i, row in resultados.reset_index(drop=True).iterrows():
        f.write(
            f"<tr>"
            f"<td>{i}</td>"
            f"<td>{row['codigo_postal']}</td>"
            f"<td>{round(row['dist_km'],2)}</td>"
            f"<td><a href='mapas/mapa_{i}.html' target='_blank'>Ver y analizar</a></td>"
            f"</tr>"
        )

    f.write("</table>")

print("✅ Listo: abre 'index.html' en tu navegador.")


Filas totales CSV: 4825
Filas con coords válidas MX: 592
Filas finales (>=10km): 592
✅ Listo: abre 'index.html' en tu navegador.
